# Part 1: Fine-tuning Cross-Encoders

## Imports

In [2]:
# pip install sentence-transformers==3.4.1 pytrec_eval ranx
from torch.utils.data import DataLoader
from sentence_transformers import util
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from sentence_transformers import InputExample
from datetime import datetime
import gzip, os, tarfile, tqdm, logging
from collections import defaultdict
import numpy as np
import pytrec_eval
import datasets

logging.basicConfig(format='%(asctime)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
logger = logging.getLogger()
logger.setLevel(logging.INFO)


c:\Users\Matiss\anaconda3\envs\py312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
W0503 18:36:25.727000 9988 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## Hyperparameters

In [3]:
train_batch_size = 32
num_epochs = 1
pos_neg_ratio = 4
max_train_samples = 5e6

# change model_name to switch between the three models:
# 'cross-encoder/ms-marco-MiniLM-L-2-v2'
# 'cross-encoder/ms-marco-TinyBERT-L-2-v2'
# 'distilroberta-base'
model_name = 'cross-encoder/ms-marco-MiniLM-L-2-v2'

# local save path (no Google Drive needed)
model_save_path = 'finetuned_models/cross-encoder-' + model_name.replace('/', '-') + '-' + datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
os.makedirs(model_save_path, exist_ok=True)
print(f"Will save to: {model_save_path}")


Will save to: finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2026-05-03_18-37-00


## Load model

In [4]:
model = CrossEncoder(model_name, num_labels=1, max_length=512)


2026-05-03 18:37:03 - Use pytorch device: cuda


## Download MS MARCO data and load training samples

In [5]:
data_folder = 'msmarco-data'
os.makedirs(data_folder, exist_ok=True)

# corpus
corpus = {}
collection_filepath = os.path.join(data_folder, 'collection.tsv')
if not os.path.exists(collection_filepath):
    tar_filepath = os.path.join(data_folder, 'collection.tar.gz')
    if not os.path.exists(tar_filepath):
        logging.info("Downloading collection.tar.gz")
        util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz', tar_filepath)
    with tarfile.open(tar_filepath, 'r:gz') as tar:
        tar.extractall(path=data_folder)

with open(collection_filepath, 'r', encoding='utf8') as f:
    for line in f:
        pid, passage = line.strip().split('\t')
        corpus[pid] = passage

# queries
queries_train = {}
queries_filepath = 'queries.train.tsv'
if not os.path.exists(queries_filepath):
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz', 'queries.tar.gz')
    with tarfile.open('queries.tar.gz', 'r:gz') as tar:
        tar.extractall()

with open(queries_filepath, 'r', encoding='utf8') as f:
    for line in f:
        qid, query = line.strip().split('\t')
        queries_train[qid] = query

# train / dev split
train_samples = []
dev_samples = {}
num_dev_queries = 200
num_max_dev_negatives = 200

train_eval_filepath = os.path.join(data_folder, 'msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz')
if not os.path.exists(train_eval_filepath):
    util.http_get('https://sbert.net/datasets/msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz', train_eval_filepath)

with gzip.open(train_eval_filepath, 'rt') as f:
    for line in f:
        qid, pos_id, neg_id = line.strip().split()
        if qid not in dev_samples and len(dev_samples) < num_dev_queries:
            dev_samples[qid] = {'query': queries_train[qid], 'positive': set(), 'negative': set()}
        if qid in dev_samples:
            dev_samples[qid]['positive'].add(corpus[pos_id])
            if len(dev_samples[qid]['negative']) < num_max_dev_negatives:
                dev_samples[qid]['negative'].add(corpus[neg_id])

train_filepath = os.path.join(data_folder, 'msmarco-qidpidtriples.rnd-shuf.train.tsv.gz')
if not os.path.exists(train_filepath):
    util.http_get('https://sbert.net/datasets/msmarco-qidpidtriples.rnd-shuf.train.tsv.gz', train_filepath)

cnt = 0
with gzip.open(train_filepath, 'rt') as f:
    for line in tqdm.tqdm(f, unit_scale=True):
        qid, pos_id, neg_id = line.strip().split()
        if qid in dev_samples:
            continue
        query = queries_train[qid]
        if (cnt % (pos_neg_ratio + 1)) == 0:
            passage, label = corpus[pos_id], 1
        else:
            passage, label = corpus[neg_id], 0
        train_samples.append(InputExample(texts=[query, passage], label=label))
        cnt += 1
        if cnt >= max_train_samples:
            break

print(f"Train samples: {len(train_samples):,}  |  Dev queries: {len(dev_samples):,}")


5.00Mit [00:20, 242kit/s] 

Train samples: 5,000,000  |  Dev queries: 200


## Train the model

Stop the cell manually after 1 hour. Note down the total number of steps from the log output — you need this for your report.

In [ ]:
# NO NEED TO RETRAIN

# import time

# class TimedTraining:
#     """Stops training after max_hours by raising an exception in the callback."""
#     def __init__(self, max_hours=1):
#         self.deadline = time.time() + max_hours * 3600
    
#     def __call__(self, score, epoch, steps):
#         if time.time() > self.deadline:
#             raise KeyboardInterrupt(f"1h reached at step {steps} — stopping.")

# train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=train_batch_size)
# evaluator = CERerankingEvaluator(dev_samples, name='train-eval')

# try:
#     model.fit(
#         train_dataloader=train_dataloader,
#         evaluator=evaluator,
#         epochs=num_epochs,
#         evaluation_steps=1000,
#         warmup_steps=5000,
#         output_path=model_save_path,
#         use_amp=True,
#         callback=TimedTraining(max_hours=1),
#     )
# except KeyboardInterrupt as e:
#     print(e)

# model.save(model_save_path)
# print(f"Model saved to {model_save_path}")


c:\Users\Matiss\anaconda3\envs\py312\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/156250 [00:00<?, ?it/s]

2026-05-03 17:09:17 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0 after 1000 steps:
2026-05-03 17:10:22 - Queries: 200 	 Positives: Min 1.0, Mean 1.1, Max 3.0 	 Negatives: Min 100.0, Mean 199.1, Max 200.0
2026-05-03 17:10:22 - MRR@10: 40.84
2026-05-03 17:10:22 - NDCG@10: 47.58
2026-05-03 17:10:22 - Save model to finetuned_models/cross-encoder-distilroberta-base-2026-05-03_17-07-05
2026-05-03 17:11:50 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0 after 2000 steps:
2026-05-03 17:12:56 - Queries: 200 	 Positives: Min 1.0, Mean 1.1, Max 3.0 	 Negatives: Min 100.0, Mean 199.1, Max 200.0
2026-05-03 17:12:56 - MRR@10: 49.31
2026-05-03 17:12:56 - NDCG@10: 56.12
2026-05-03 17:12:56 - Save model to finetuned_models/cross-encoder-distilroberta-base-2026-05-03_17-07-05
2026-05-03 17:14:24 - CERerankingEvaluator: Evaluating the model on train-eval dataset in epoch 0 after 3000 steps:
2026-05-03 17:15:29 - Queries: 200 	 Positives: Min 

1h reached at step 24000 — stopping.
Model saved to finetuned_models/cross-encoder-distilroberta-base-2026-05-03_17-07-05


## Evaluation on TREC DL'19

Evaluate the fine-tuned model on the 43 TREC DL'19 queries. Run this section for each of the three fine-tuned models by updating `model_save_path`.

In [6]:
# load TREC DL'19 data
DATA_DIR = "Data"  # same folder as used in tasks 4 & 5

QUERIES_PATH    = os.path.join(DATA_DIR, "queries.tsv")
CANDIDATES_PATH = os.path.join(DATA_DIR, "candidates.tsv")
QRELS_PATH      = os.path.join(DATA_DIR, "qrels.txt")

queries = {}
with open(QUERIES_PATH, 'r') as f:
    for line in f:
        qid, query = line.strip().split('\t')
        queries[qid] = query

candidates = {}
with open(CANDIDATES_PATH, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        qid, docid, doctext = parts[0], parts[1], parts[3]
        if qid not in candidates:
            candidates[qid] = []
        candidates[qid].append((docid, doctext))

qrels = {}
with open(QRELS_PATH, 'r') as f:
    for line in f:
        parts = line.strip().split()
        qid, docid, relevance = parts[0], parts[2], int(parts[3])
        if qid not in qrels:
            qrels[qid] = {}
        qrels[qid][docid] = relevance

# filter to the 43 queries with qrels
qrel_ids = set(qrels.keys())
queries    = {k: v for k, v in queries.items()    if k in qrel_ids}
candidates = {k: v for k, v in candidates.items() if k in qrel_ids}

assert len(queries) == len(candidates) == len(qrels) == 43
print("Data loaded: 43 queries")


Data loaded: 43 queries


In [13]:
model_paths = {
    "MiniLM-L2":     "finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2026-05-03_14-43-59",
    "TinyBERT-L2":   "finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2026-05-03_16-03-29",
    "DistilRoBERTa": "finetuned_models/cross-encoder-distilroberta-base-2026-05-03_17-07-05",
}

def score_candidates(queries, candidates, cross_encoder, batch_size=32):
    results = {}
    for qid, query_text in tqdm.tqdm(queries.items(), desc="Scoring"):
        docs  = candidates[qid]
        pairs = [(query_text, doc_text) for _, doc_text in docs]
        scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        results[qid] = {docid: float(s) for (docid, _), s in zip(docs, scores)}
    return results

all_scores = {}
for name, path in model_paths.items():
    eval_model = CrossEncoder(path, max_length=512, device='cuda')
    all_scores[name] = score_candidates(queries, candidates, eval_model)
    print(f"Done: {name}")

Scoring: 100%|██████████| 43/43 [00:12<00:00,  3.44it/s]


Done: MiniLM-L2


Scoring: 100%|██████████| 43/43 [00:12<00:00,  3.46it/s]


Done: TinyBERT-L2


Scoring: 100%|██████████| 43/43 [01:03<00:00,  1.48s/it]

Done: DistilRoBERTa


In [14]:
def evaluate(scores, qrels):
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10', 'recall_100', 'map_cut_1000'})
    results   = evaluator.evaluate(scores)
    metrics   = {}
    for measure in ['ndcg_cut_10', 'recall_100', 'map_cut_1000']:
        metrics[measure] = round(sum(r[measure] for r in results.values()) / len(results), 4)
    return metrics

print(f"{'Model':<15} {'NDCG@10':>10} {'Recall@100':>12} {'MAP@1000':>10}")
print('-' * 50)

for name, (path, scores) in zip(model_paths.keys(), zip(model_paths.values(), all_scores.values())):
    # save ranking.run for tasks 2 & 3
    run_file = os.path.join(path, 'ranking.run')
    with open(run_file, 'w') as f:
        for qid, pid_scores in scores.items():
            for rank, (pid, score) in enumerate(
                    sorted(pid_scores.items(), key=lambda x: x[1], reverse=True), 1):
                f.write(f"{qid} Q0 {pid} {rank} {score:.6f} cross-encoder\n")

    metrics = evaluate(scores, qrels)
    print(f"{name:<15} {metrics['ndcg_cut_10']:>10} {metrics['recall_100']:>12} {metrics['map_cut_1000']:>10}")

Model              NDCG@10   Recall@100   MAP@1000
--------------------------------------------------
MiniLM-L2           0.6724       0.4985     0.4293
TinyBERT-L2         0.6703       0.4973      0.443
DistilRoBERTa       0.6599       0.4907     0.4227


# Tasks 2 & 3: Ensemble Methods

Load the three `ranking.run` files produced above and apply fusion methods using ranx.

In [19]:
from ranx import Qrels, Run, fuse, evaluate as ranx_evaluate

run_paths = {
    "MiniLM-L2":     "finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2026-05-03_14-43-59/ranking.run",
    "TinyBERT-L2":   "finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2026-05-03_16-03-29/ranking.run",
    "DistilRoBERTa": "finetuned_models/cross-encoder-distilroberta-base-2026-05-03_17-07-05/ranking.run",
}

ranx_runs  = [Run.from_file(path, kind="trec", name=name) for name, path in run_paths.items()]
ranx_qrels = Qrels.from_file(QRELS_PATH, kind="trec")

## Task 2: Five fusion methods

In [ ]:
FUSION_METHODS = ['sum', 'mnz', 'rrf', 'bordafuse', 'condorcet']
print(f"{'Method':<15} {'NDCG@10':>10} {'Recall@100':>12} {'MAP@1000':>10}")
print('-' * 50)
for method in FUSION_METHODS:
    fused  = fuse(runs=ranx_runs, method=method)
    scores = ranx_evaluate(ranx_qrels, fused, ['ndcg@10', 'recall@100', 'map@1000'])
    print(f"{method:<15} {scores['ndcg@10']:>10.4f} {scores['recall@100']:>12.4f} {scores['map@1000']:>10.4f}")


Method             NDCG@10   Recall@100   MAP@1000
--------------------------------------------------
sum                 0.6978       0.5095     0.4567
mnz                 0.6978       0.5095     0.4567
rrf                 0.7053       0.5088     0.4600
bordafuse           0.7087       0.5068     0.4597
condorcet           0.7004       0.5088     0.4576


## Task 3: Best method on pairwise model combinations

In [ ]:
from itertools import combinations

best_method = 'bordafuse'  # update to the best method from above listed results

print(f"Fusion method: {best_method}")
print(f"{'Combination':<45} {'NDCG@10':>10} {'Recall@100':>12} {'MAP@1000':>10}")
print('-' * 80)

names = list(run_paths.keys())
for (i, run_a), (j, run_b) in combinations(enumerate(ranx_runs), 2):
    label  = f"{names[i]} + {names[j]}"
    fused  = fuse(runs=[run_a, run_b], method=best_method)
    scores = ranx_evaluate(ranx_qrels, fused, ['ndcg@10', 'recall@100', 'map@1000'])
    print(f"{label:<45} {scores['ndcg@10']:>10.4f} {scores['recall@100']:>12.4f} {scores['map@1000']:>10.4f}")


Fusion method: bordafuse
Combination                                      NDCG@10   Recall@100   MAP@1000
--------------------------------------------------------------------------------
MiniLM-L2 + TinyBERT-L2                           0.6872       0.5004     0.4509
MiniLM-L2 + DistilRoBERTa                         0.7012       0.5046     0.4488
TinyBERT-L2 + DistilRoBERTa                       0.6966       0.5044     0.4584
